# Experiment

## Import libraries

In [143]:
import pandas as pd

In [144]:
import os
import pandas as pd
from glob import glob

# Path to directory containing CSV files
data_dir = "./"  # <-- Change this to your actual folder path
file_pattern = os.path.join(data_dir, "vietstock_*.csv")
files = sorted(glob(file_pattern))

# Mapping Vietnamese column names to English
rename_map = {
    "1 tuần": "1_week",
    "2 tuần": "2_week",
    "1 tháng": "1_month",
    "3 tháng": "3_month",
    "6 tháng": "6_month",
    "9 tháng": "9_month",
    # Remove 'Qua đêm' so we won't rename, but drop later
}

combined_data = []

for file in files:
    year = os.path.basename(file).split("_")[1].split(".")[0]
    df = pd.read_csv(file, index_col=0)

    # Skip files with only headers or no data
    if df.shape[1] <= 3:
        print(f"Skipping {file} (likely no actual data)")
        continue

    # Clean column names and rows
    df = df.drop(columns=["Đơn vị tính"], errors="ignore")
    df = df.dropna(how="all", axis=1)  # Drop completely empty columns
    df = df.dropna(how="all", axis=0)  # Drop completely empty rows

    df = df.T  # Transpose to have dates as rows
    df["date"] = df.index
    df['date'] = pd.to_datetime(df['date'], dayfirst=True)
    df.reset_index(drop=True, inplace=True)

    combined_data.append(df)

# Merge all into one big DataFrame
result_df = pd.concat(combined_data)

# Reset index to remove old index and get clean integer index
result_df = result_df.reset_index(drop=True)

# Sort by date
result_df = result_df.sort_values(by="date")

# Reorder columns to have 'date' first
cols = result_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('date')))
result_df = result_df[cols]
result_df = result_df.drop(columns=["Qua đêm"], errors="ignore")
result_df = result_df.rename(columns=rename_map)

# Optional: Save to CSV without the index
result_df.to_csv("combined_interest_rates.csv", index=False)

# Display sample
result_df


Skipping .\vietstock_2000.csv (likely no actual data)
Skipping .\vietstock_2001.csv (likely no actual data)
Skipping .\vietstock_2002.csv (likely no actual data)
Skipping .\vietstock_2003.csv (likely no actual data)


Chỉ tiêu,date,1_week,2_week,1_month,3_month,6_month,9_month
0,2004-07-12,6.05,6.23,6.58,6.93,7.21,0.00
1,2004-07-14,6.16,6.37,6.71,7.09,7.31,0.00
2,2004-07-19,6.14,6.36,6.67,7.04,7.53,0.00
3,2004-07-21,6.17,6.32,6.67,6.98,7.64,0.00
4,2004-07-26,5.97,6.04,6.42,6.84,7.22,0.00
...,...,...,...,...,...,...,...
5121,2025-05-13,4.22,4.30,4.32,4.65,4.88,4.46
5122,2025-05-14,4.08,4.25,4.15,4.67,4.88,4.46
5123,2025-05-15,4.08,4.16,4.33,4.68,4.88,4.46
5124,2025-05-16,3.92,4.13,4.13,4.62,4.88,4.46
